# Part C — Vector Spaces

**Final Project: Philippine Peso Depreciation Early-Warning Model: Predicting the Next-Month PHP/USD Exchange Rate Using Linear Regression and PCA**

The columns of X don't just sit there — together they define a subspace that limits what your model can ever predict.

---
**Student name:** Alfajaro, Ghan Vincent | Arile, Charles Jullianne | Asuar, Jhoe Maven | Mondragon, Leneth | Ulitin, Justin |   **Date:** November 16, 2026

---


**What to do:** Compute the rank of X. Determine whether its columns are linearly independent.

In [1]:
# Run this first: recall variables saved from the previous part's notebook.
# (If this is your first notebook in the session, or the store is empty,
#  you'll need to re-run/paste the relevant setup code from the earlier part.)
%store -r X XtX

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

In [3]:
# Compute the rank of your design matrix X
rank_X = np.linalg.matrix_rank(X)
n_samples, n_features = X.shape
print(f"X is {n_samples} x {n_features}.  rank(X) = {rank_X}")

X is 428 x 3.  rank(X) = 3


In [4]:
# Compare rank(X) to the number of features.
# Are the columns of X linearly independent? Print a short conclusion.
if rank_X == n_features:
    print(f"rank(X) = {rank_X} = p = {n_features}: full column rank.")
    print("The columns of X are linearly independent.")
else:
    print(f"rank(X) = {rank_X} < p = {n_features}: rank deficient.")
    print("At least one column is a linear combination of the others.")

# Extra check: independence is exact, but how close are we to losing it?
sv = np.linalg.svd(X, compute_uv=False)
print("\nSingular values of X:", sv)
print(f"Condition number of X: {sv[0] / sv[-1]:.2f}")

# Extra check: what would a dependent feature do to the rank?
X_dep = np.column_stack([X, 3 * X[:, 1]])   # add 3 * Trailing3MonthAverage as a 4th column
print("\nWith a redundant 4th column:")
print("X_dep shape:", X_dep.shape, "| rank(X_dep) =", np.linalg.matrix_rank(X_dep))
print("rank of X_dep^T X_dep =", np.linalg.matrix_rank(X_dep.T @ X_dep), "(4x4, so it is singular)")

rank(X) = 3 = p = 3: full column rank.
The columns of X are linearly independent.

Singular values of X: [1344.4382   75.8569    8.4535]
Condition number of X: 159.04

With a redundant 4th column:
X_dep shape: (428, 4) | rank(X_dep) = 3
rank of X_dep^T X_dep = 3 (4x4, so it is singular)


### Answer these in your report

**Q9.** What is the dimension of the feature space your data lives in (before any reduction)?

> *Your answer:* Our data lives in a 3-dimensional feature space, ℝ³, because we have p = 3 features. Each month is a single point in that space, with coordinates given by its current rate, trailing three-month average, and year-over-year change. The 428 months are 428 points scattered in that 3D space. The columns of X are separate vectors that live in ℝ⁴²⁸, one entry per month, and that is the space where the column space sits.

**Q10.** What is rank(X)? Is it equal to the number of features p? If not, what does that tell you about your features?

> *Your answer:* Our code gives rank(X) = 3, which equals p = 3, so X has full column rank. This means the three features are linearly independent: none of them can be written exactly as a combination of the others. That is a good sign for the regression, because XᵀX is invertible and the normal equations have exactly one solution for β. Independence does not mean the features are unrelated, though. X1 is part of how X2 is built, so those two columns are very close to moving together. The condition number we printed shows how close we are to losing independence. A large value means the system is technically solvable but sensitive, so small changes in the data could shift the coefficients noticeably.

**Q11.** In your own words, what is the column space of X, and why does it matter for what predictions your regression model can possibly make?

> *Your answer:* The column space of X is the set of every vector we can get by mixing the three columns, which is every Xβ for some choice of β. Since there are three independent columns, it is a flat 3-dimensional subspace sitting inside the 428-dimensional space of monthly values. This matters because a linear regression can only output predictions of the form ŷ = Xβ, so every prediction we make is forced to lie in this subspace. The real next-month rates in y almost certainly do not lie exactly inside it, since the peso has movements that our three features cannot capture. So the best the model can do is find the point in the column space that is closest to y, which is the projection of y onto that subspace. The leftover gap is the residual, and it is perpendicular to the column space.

**Q12.** Describe what a linearly dependent feature would look like in your own dataset (a real or hypothetical example), and why it would be a problem for your model.

> *Your answer:* A hypothetical example in our dataset is adding a fourth feature called "sum of the last three monthly rates." This is just 3 times the trailing three-month average (X2), so the new column is an exact multiple of an existing one. The rank would stay at 3 even though we now have 4 columns, and XᵀX would become a singular 4 × 4 matrix with no inverse. That is a problem because the normal equations would have infinitely many solutions. The model could put weight on X2 or on the copy of X2 in countless ways that give identical predictions, so the individual coefficients would stop meaning anything. In practice, even a feature that is nearly but not exactly redundant causes unstable coefficients, which is why we watch how closely X1 and X2 track each other.

### Save your work for the next notebook

In [5]:
%store rank_X
print("Saved rank_X for Part D.")

Stored 'rank_X' (int64)
Saved rank_X for Part D.
